In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

In [4]:
# Iniciar uma sessão Spark
spark = SparkSession.builder \
    .appName("ExemploSparkSQL") \
    .getOrCreate()

In [5]:
# Criar um DataFrame de exemplo para vendas
vendas_data = [
    ("2023-01-01", "Produto A", 100, 10.5),
    ("2023-01-02", "Produto B", 150, 20.0),
    ("2023-01-03", "Produto A", 200, 10.5),
    ("2023-01-04", "Produto C", 50, 15.0),
    ("2023-01-05", "Produto B", 300, 20.0)
]
vendas_df = spark.createDataFrame(vendas_data, ["data", "produto", "quantidade", "preco_unitario"])

# Criar um DataFrame de exemplo para produtos
produtos_data = [
    ("Produto A", "Eletrônicos"),
    ("Produto B", "Roupas"),
    ("Produto C", "Alimentos")
]
produtos_df = spark.createDataFrame(produtos_data, ["produto", "categoria"])

# Registrar os DataFrames como tabelas temporárias
vendas_df.createOrReplaceTempView("vendas")
produtos_df.createOrReplaceTempView("produtos")

# Executar uma consulta SQL
resultado = spark.sql("""
    SELECT
        v.data,
        v.produto,
        p.categoria,
        v.quantidade,
        v.preco_unitario,
        v.quantidade * v.preco_unitario as valor_total
    FROM vendas v
    JOIN produtos p ON v.produto = p.produto
    ORDER BY v.data
""")

print("Resultado da consulta SQL:")
resultado.show()

Resultado da consulta SQL:
+----------+---------+-----------+----------+--------------+-----------+
|      data|  produto|  categoria|quantidade|preco_unitario|valor_total|
+----------+---------+-----------+----------+--------------+-----------+
|2023-01-01|Produto A|Eletrônicos|       100|          10.5|     1050.0|
|2023-01-02|Produto B|     Roupas|       150|          20.0|     3000.0|
|2023-01-03|Produto A|Eletrônicos|       200|          10.5|     2100.0|
|2023-01-04|Produto C|  Alimentos|        50|          15.0|      750.0|
|2023-01-05|Produto B|     Roupas|       300|          20.0|     6000.0|
+----------+---------+-----------+----------+--------------+-----------+



In [6]:
# Calcular o total de vendas por categoria usando SQL
total_por_categoria = spark.sql("""
    SELECT
        p.categoria,
        SUM(v.quantidade * v.preco_unitario) as total_vendas
    FROM vendas v
    JOIN produtos p ON v.produto = p.produto
    GROUP BY p.categoria
    ORDER BY total_vendas DESC
""")

print("Total de vendas por categoria:")
total_por_categoria.show()

Total de vendas por categoria:
+-----------+------------+
|  categoria|total_vendas|
+-----------+------------+
|     Roupas|      9000.0|
|Eletrônicos|      3150.0|
|  Alimentos|       750.0|
+-----------+------------+



In [7]:
# Exemplo de uso de funções do Spark SQL
media_quantidade = vendas_df.select(avg("quantidade").alias("media_quantidade"))
print("Média de quantidade vendida:")
media_quantidade.show()

Média de quantidade vendida:
+----------------+
|media_quantidade|
+----------------+
|           160.0|
+----------------+



In [8]:
# Exemplo de filtro usando DataFrame API
vendas_altas = vendas_df.filter(col("quantidade") > 100)
print("Vendas com quantidade maior que 100:")
vendas_altas.show()

# Parar a sessão Spark
spark.stop()

Vendas com quantidade maior que 100:
+----------+---------+----------+--------------+
|      data|  produto|quantidade|preco_unitario|
+----------+---------+----------+--------------+
|2023-01-02|Produto B|       150|          20.0|
|2023-01-03|Produto A|       200|          10.5|
|2023-01-05|Produto B|       300|          20.0|
+----------+---------+----------+--------------+

